<a href="https://colab.research.google.com/github/jihoon0915-gif/-AI-BIGDATA-34-/blob/main/7_Backpropagation_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Backpropagation**
Work through the cells below, running each cell in turn. In various places you will see the words "TO DO". Follow the instructions at these places and make predictions about what is going to happen or write code to complete the functions.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

First let's define a neural network.  We'll just choose the weights and biases randomly for now

In [2]:
# Set seed so we always get the same random numbers
np.random.seed(0)

# Number of layers
K = 5
# Number of neurons per layer
D = 6
# Input layer
D_i = 1
# Output layer
D_o = 1

# Make empty lists
all_weights = [None] * (K+1)
all_biases = [None] * (K+1)

# Create input and output layers
all_weights[0] = np.random.normal(size=(D, D_i))
all_weights[-1] = np.random.normal(size=(D_o, D))
all_biases[0] = np.random.normal(size =(D,1))
all_biases[-1]= np.random.normal(size =(D_o,1))

# Create intermediate layers
for layer in range(1,K):
  all_weights[layer] = np.random.normal(size=(D,D))
  all_biases[layer] = np.random.normal(size=(D,1))

![image.png](attachment:image.png)

In [3]:
# Define the Rectified Linear Unit (ReLU) function
def ReLU(preactivation):
  activation = preactivation.clip(0.0)
  return activation

Now let's run the forward pass


In [4]:
def compute_network_output(net_input, all_weights, all_biases):

  # Retrieve number of layers
  K = len(all_weights) -1

  # We'll store the pre-activations at each layer in a list "all_f"
  # and the activations in a second list "all_h".
  all_f = [None] * (K+1)
  all_h = [None] * (K+1)

  #For convenience, we'll set
  # all_h[0] to be the input, and all_f[K] will be the output
  all_h[0] = net_input

  # Run through the layers, calculating all_f[0...K-1] and all_h[1...K]
  for layer in range(K):
      # Update preactivations and activations at this layer
      # Remember to use np.matmul for matrix multiplications
      # TODO -- Replace the lines below
      all_f[layer] = all_h[layer]
      all_h[layer+1] = all_f[layer]

  # Compute the output from the last hidden layer
  # TODO -- Replace the line below
  all_f[K] = np.zeros_like(all_biases[-1])

  # Retrieve the output
  net_output = all_f[K]

  return net_output, all_f, all_h

In [5]:
# Define input
net_input = np.ones((D_i,1)) * 1.2
# Compute network output
net_output, all_f, all_h = compute_network_output(net_input,all_weights, all_biases)
print("True output = %3.3f, Your answer = %3.3f"%(1.907, net_output[0,0]))

True output = 1.907, Your answer = 0.000


Now let's define a loss function.  We'll just use the least squares loss function. We'll also write a function to compute dloss_doutput

In [6]:
def least_squares_loss(net_output, y):
  return np.sum((net_output-y) * (net_output-y))

def d_loss_d_output(net_output, y):
    return 2*(net_output -y);

In [7]:
y = np.ones((D_o,1)) * 20.0
loss = least_squares_loss(net_output, y)
print("y = %3.3f Loss = %3.3f"%(y, loss))

y = 20.000 Loss = 400.000


/tmp/ipykernel_2433/3361061389.py:3: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  print("y = %3.3f Loss = %3.3f"%(y, loss))


Now let's compute the derivatives of the network.  We already computed the forward pass.  Let's compute the backward pass.

### Backpropagation

**1. Output에서 시작**

$$
\boxed{
\frac{\partial L}{\partial f^{[K]}}
=
\frac{\partial L}{\partial \hat{y}}
=
2(\hat{y}-y)
}
$$

---

**2. Bias Gradient**

$$
\frac{\partial L}{\partial b^{[l]}}
=
\frac{\partial L}{\partial f^{[l]}}
\frac{\partial f^{[l]}}{\partial b^{[l]}}
$$

$$
\frac{\partial f^{[l]}}{\partial b^{[l]}}=1
$$

따라서

$$
\boxed{
\frac{\partial L}{\partial b^{[l]}}
=
\frac{\partial L}{\partial f^{[l]}}
}
$$

---

**3. Weight Gradient**

$$
\frac{\partial L}{\partial W^{[l]}}
=
\frac{\partial L}{\partial f^{[l]}}
\frac{\partial f^{[l]}}{\partial W^{[l]}}
$$

Forward에서

$$
f^{[l]}=W^{[l]}h^{[l]}+b^{[l]}
$$

이므로

$$
\frac{\partial f^{[l]}}{\partial W^{[l]}}
\rightarrow (h^{[l]})^T
$$

따라서

$$
\boxed{
\frac{\partial L}{\partial W^{[l]}}
=
\frac{\partial L}{\partial f^{[l]}}
(h^{[l]})^T
}
$$

---

**4. Activation Gradient**

$$
\frac{\partial L}{\partial h^{[l]}}
=
\frac{\partial f^{[l]}}{\partial h^{[l]}}
\frac{\partial L}{\partial f^{[l]}}
$$

Forward에서

$$
f^{[l]}=W^{[l]}h^{[l]}+b^{[l]}
$$

이므로

$$
\frac{\partial f^{[l]}}{\partial h^{[l]}}
=
W^{[l]}
$$

행렬의 방향을 맞추면

$$
\boxed{
\frac{\partial L}{\partial h^{[l]}}
=
(W^{[l]})^T
\frac{\partial L}{\partial f^{[l]}}
}
$$

---

**5. ReLU를 통과하여 이전 Layer로**

Forward에서

$$
h^{[l]}
=
\mathrm{ReLU}(f^{[l-1]})
$$

이므로 Chain Rule에 의해

$$
\frac{\partial L}{\partial f^{[l-1]}}
=
\frac{\partial L}{\partial h^{[l]}}
\odot
\frac{\partial h^{[l]}}{\partial f^{[l-1]}}
$$

그리고

$$
\frac{\partial h^{[l]}}{\partial f^{[l-1]}}
=
\mathrm{ReLU}'(f^{[l-1]})
$$

따라서

$$
\boxed{
\frac{\partial L}{\partial f^{[l-1]}}
=
\frac{\partial L}{\partial h^{[l]}}
\odot
\mathrm{ReLU}'(f^{[l-1]})
}
$$

---

### 전체 Backpropagation 흐름

$$
\boxed{
\frac{\partial L}{\partial f^{[K]}}
\;\longrightarrow\;
\left(
\frac{\partial L}{\partial b^{[l]}},
\frac{\partial L}{\partial W^{[l]}}
\right)
\;\longrightarrow\;
\frac{\partial L}{\partial h^{[l]}}
\;\longrightarrow\;
\frac{\partial L}{\partial f^{[l-1]}}
\;\longrightarrow\;
\text{Repeat}
}
$$

In [8]:
# We'll need the indicator function
def indicator_function(x):
  x_in = np.array(x)
  x_in[x_in>=0] = 1
  x_in[x_in<0] = 0
  return x_in

# Main backward pass routine
def backward_pass(all_weights, all_biases, all_f, all_h, y):
  # We'll store the derivatives dl_dweights and dl_dbiases in lists as well
  all_dl_dweights = [None] * (K+1)
  all_dl_dbiases = [None] * (K+1)
  # And we'll store the derivatives of the loss with respect to the activation and preactivations in lists
  all_dl_df = [None] * (K+1)
  all_dl_dh = [None] * (K+1)
  # Again for convenience we'll stick with the convention that all_h[0] is the net input and all_f[k] in the net output

  # Compute derivatives of the loss with respect to the network output
  all_dl_df[K] = np.array(d_loss_d_output(all_f[K],y))

  # Now work backwards through the network
  for layer in range(K,-1,-1):
    # TODO Calculate the derivatives of the loss with respect to the biases at layer from all_dl_df[layer].
    # NOTE!  To take a copy of matrix X, use Z=np.array(X)
    # REPLACE THIS LINE
    all_dl_dbiases[layer] = np.zeros_like(all_biases[layer])

    # TODO Calculate the derivatives of the loss with respect to the weights at layer from all_dl_df[layer] and all_h[layer]
    # Don't forget to use np.matmul
    # REPLACE THIS LINE
    all_dl_dweights[layer] = np.zeros_like(all_weights[layer])

    # TODO: calculate the derivatives of the loss with respect to the activations from weight and derivatives of next preactivations
    # REPLACE THIS LINE
    all_dl_dh[layer] = np.zeros_like(all_h[layer])


    if layer > 0:
      # TODO Calculate the derivatives of the loss with respect to the pre-activation f (use derivative of ReLu function)
      # REPLACE THIS LINE
      all_dl_df[layer-1] = np.zeros_like(all_f[layer-1])

  return all_dl_dweights, all_dl_dbiases

In [9]:
all_dl_dweights, all_dl_dbiases = backward_pass(all_weights, all_biases, all_f, all_h, y)

In [10]:
np.set_printoptions(precision=3)
# Make space for derivatives computed by finite differences
all_dl_dweights_fd = [None] * (K+1)
all_dl_dbiases_fd = [None] * (K+1)

# Let's test if we have the derivatives right using finite differences
delta_fd = 0.000001

# Test the dervatives of the bias vectors
for layer in range(K):
  dl_dbias  = np.zeros_like(all_dl_dbiases[layer])
  # For every element in the bias
  for row in range(all_biases[layer].shape[0]):
    # Take copy of biases  We'll change one element each time
    all_biases_copy = [np.array(x) for x in all_biases]
    all_biases_copy[layer][row] += delta_fd
    network_output_1, *_ = compute_network_output(net_input, all_weights, all_biases_copy)
    network_output_2, *_ = compute_network_output(net_input, all_weights, all_biases)
    dl_dbias[row] = (least_squares_loss(network_output_1, y) - least_squares_loss(network_output_2,y))/delta_fd
  all_dl_dbiases_fd[layer] = np.array(dl_dbias)
  print("-----------------------------------------------")
  print("Bias %d, derivatives from backprop:"%(layer))
  print(all_dl_dbiases[layer])
  print("Bias %d, derivatives from finite differences"%(layer))
  print(all_dl_dbiases_fd[layer])
  if np.allclose(all_dl_dbiases_fd[layer],all_dl_dbiases[layer],rtol=1e-05, atol=1e-08, equal_nan=False):
    print("Success!  Derivatives match.")
  else:
    print("Failure!  Derivatives different.")



# Test the derivatives of the weights matrices
for layer in range(K):
  dl_dweight  = np.zeros_like(all_dl_dweights[layer])
  # For every element in the bias
  for row in range(all_weights[layer].shape[0]):
    for col in range(all_weights[layer].shape[1]):
      # Take copy of biases  We'll change one element each time
      all_weights_copy = [np.array(x) for x in all_weights]
      all_weights_copy[layer][row][col] += delta_fd
      network_output_1, *_ = compute_network_output(net_input, all_weights_copy, all_biases)
      network_output_2, *_ = compute_network_output(net_input, all_weights, all_biases)
      dl_dweight[row][col] = (least_squares_loss(network_output_1, y) - least_squares_loss(network_output_2,y))/delta_fd
  all_dl_dweights_fd[layer] = np.array(dl_dweight)
  print("-----------------------------------------------")
  print("Weight %d, derivatives from backprop:"%(layer))
  print(all_dl_dweights[layer])
  print("Weight %d, derivatives from finite differences"%(layer))
  print(all_dl_dweights_fd[layer])
  if np.allclose(all_dl_dweights_fd[layer],all_dl_dweights[layer],rtol=1e-05, atol=1e-08, equal_nan=False):
    print("Success!  Derivatives match.")
  else:
    print("Failure!  Derivatives different.")

-----------------------------------------------
Bias 0, derivatives from backprop:
[[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]
Bias 0, derivatives from finite differences
[[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]
Success!  Derivatives match.
-----------------------------------------------
Bias 1, derivatives from backprop:
[[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]
Bias 1, derivatives from finite differences
[[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]
Success!  Derivatives match.
-----------------------------------------------
Bias 2, derivatives from backprop:
[[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]
Bias 2, derivatives from finite differences
[[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]
Success!  Derivatives match.
-----------------------------------------------
Bias 3, derivatives from backprop:
[[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]
Bias 3, derivatives from finite differences
[[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]
Success!  Derivatives match.
-----------------------------------------------
Bias 4, derivatives from backpro